# Invervention Flags

Identified Contaminated Parks (parks that have timed entries)


### Step 1: Load cleaned data 

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt


# plotting style
plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1.5,
    'figure.figsize' : [10.0, 3.0],
    'figure.dpi' :300,
    'image.aspect': 7 
})

In [6]:
# Upload the data

DATA_PATH = '../data/'
FIGURE_PATH = '../figures/'

RAW_FILE =  DATA_PATH+'raw/Main_Data.xls'
PARKS_FILE =  DATA_PATH+'raw/national_parks_lookup.csv'

VISITS_FILE =  DATA_PATH+'processed/monthly_visitation.csv'

In [7]:
df = pd.read_csv(VISITS_FILE)

df.head()

,park_name,park_code,year,month,visits
0,Pinnacles National Park,PINN,2007,1,10006
1,Pinnacles National Park,PINN,2007,2,11119
2,Pinnacles National Park,PINN,2007,3,17662
3,Pinnacles National Park,PINN,2007,4,30226
4,Pinnacles National Park,PINN,2007,5,15504


### Step 2: Create park-level dataframe 

In [8]:
parks = (df[["park_name","park_code"]].drop_duplicates().reset_index(drop=True))

parks.head()

,park_name,park_code
0,Pinnacles National Park,PINN
1,Dry Tortugas National Park,DRTO
2,Rocky Mountain National Park,ROMO
3,Congaree National Park,CONG
4,Badlands National Park,BADL


### Step 3: Create invervention database 
Manually define major visitor-management interventions.

Remove parks where a reservation system could affect total monthly recreation visitation counts (park entry, vehicle entry, or major access corridor restrictions). We did not remove individual trail/cave permits unless they substantially control park access.

In [9]:
interventions = pd.DataFrame(
    {
        "park_code": [
            "ROMO",
            "ARCH",
            "GLAC",
            "YOSE",
            "MORA",
            "ACAD",
            "HALE",
        ],

        "intervention_type": [
            "timed_entry",
            "timed_entry",
            "vehicle_reservation",
            "vehicle_reservation",
            "timed_entry",
            "vehicle_reservation",
            "vehicle_reservation",
        ],

        "start_year": [
            2020,
            2022,
            2021,
            2020,
            2024,
            2021,
            2017,
        ]
    }
)

### Step 4: Merge 

In [ ]:
parks = parks.merge(
    interventions,
    on="park_code",
    how="left"
)

In [11]:
parks

,park_name,park_code,intervention_type,start_year
0,Pinnacles National Park,PINN,NaN,NaN
1,Dry Tortugas National Park,DRTO,NaN,NaN
2,Rocky Mountain National Park,ROMO,timed_entry,2020.0
3,Congaree National Park,CONG,NaN,NaN
4,Badlands National Park,BADL,NaN,NaN
...,...,...,...,...
57,Arches National Park,ARCH,timed_entry,2022.0
58,Isle Royale National Park,ISRO,NaN,NaN
59,Channel Islands National Park,CHIS,NaN,NaN
60,Lake Clark National Park,LACL,NaN,NaN


### Step 5: Fill untreated parks

In [12]:
parks["intervention_type"] = (
    parks["intervention_type"]
    .fillna("none")
)

parks["timed_entry"] = (
    parks["intervention_type"] != "none"
).astype(int)

### Step 6: Define donor eligibility 

In [15]:
parks["donor_eligible"] = 1

In [16]:
# Remove RMNP 
parks.loc[parks["park_code"] == "ROMO", "donor_eligible"] = 0

In [17]:
# Remove contaminated controls
parks.loc[parks["timed_entry"] == 1,"donor_eligible"] = 0

### Step 7: Check donor pool

In [18]:
parks[parks["donor_eligible"] == 1].shape

(55, 6)

### Step 8: Save the resulting file 

In [19]:
filename = 'park_intervention_flags.csv' 
parks.to_csv(DATA_PATH + 'processed/' + filename, index = False)